In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc
import pandas as pd
from matplotlib.widgets import Slider
from skimage.transform import resize, rotate
import matplotlib.image as mpimg
from matplotlib.widgets import Slider, CheckButtons

In [2]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        if self.image.ndim == 2:
            ny, nx = self.image.shape
        else:
            ny, nx = self.image.shape[:2]
        y, x = np.mgrid[:ny, :nx]

        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [3]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [ ]:
# Load and visualize the m/z heatmap from AnnData
input_file = "processed_h5ad_file_address.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/h5ad_data_processed/OFM_AD_3_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [5]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [6]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
batch = adata.obs['batch'][0] 
sample = adata.obs['sample'][0]
adata.var

/tmp/ipykernel_3314502/3761597262.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  batch = adata.obs['batch'][0]
/tmp/ipykernel_3314502/3761597262.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample = adata.obs['sample'][0]


,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [7]:
def load_he_image(path, target_shape=None):
    """
    Reads and resizes an H&E image.

    Parameters:
        path (str): Path to the H&E image file.
        target_shape (tuple): (height, width) to resize the image to match MSI.

    Returns:
        np.ndarray: RGB image resized and normalized.
    """
    img = mpimg.imread(path)

    # Resize if a target shape is provided
    if target_shape is not None:
        img = resize(img, target_shape, preserve_range=True)

    # Ensure RGB format
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)

    # Normalize to [0, 1]
    img = np.clip(img / 255.0 if img.max() > 1 else img, 0, 1)
    return img

In [ ]:

# ---- Run ----

#target_mz = 788.5
target_mz = 800.55


image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

# Normalize MSI and convert to RGB
heatmap_norm = (image - np.min(image)) / (np.max(image) - np.min(image))
heatmap_rgb = custom_cmap(heatmap_norm)[..., :3]  # Drop alpha

# Load original image (without resizing)
he_image_original = load_he_image("sample_image_address.jpg")

# Transformation parameters
init_angle = 0
init_scale_x = 1.0
init_scale_y = 1.0
init_tx = 0
init_ty = 0
init_alpha = 0.5

# Plot setup
fig, ax = plt.subplots()
plt.subplots_adjust(bottom=0.45)

# Initial empty image
img_ax = ax.imshow(np.zeros_like(image), cmap='gray')

# Overlay update function
def update_overlay(angle, scale_x, scale_y, tx, ty, alpha, uniform):
    # Uniform scaling override
    if uniform:
        scale_y = scale_x

    # Transform
    rotated = rotate(he_image_original, angle, resize=False, preserve_range=True)
    target_shape = (
        int(image.shape[0] * scale_y),
        int(image.shape[1] * scale_x)
    )
    resized = resize(rotated, target_shape, preserve_range=True)

    # Translation
    out_image = np.zeros_like(heatmap_rgb)
    h, w = out_image.shape[:2]
    rh, rw = resized.shape[:2]

    # Compute placement bounds
    y_start = int((h - rh) / 2 + ty)
    x_start = int((w - rw) / 2 + tx)

    # Insert transformed image within bounds
    y_end = min(y_start + rh, h)
    x_end = min(x_start + rw, w)

    ys = max(-y_start, 0)
    xs = max(-x_start, 0)

    if y_start < h and x_start < w:
        out_image[
            max(y_start, 0):y_end,
            max(x_start, 0):x_end
        ] = resized[ys:ys + (y_end - max(y_start, 0)), xs:xs + (x_end - max(x_start, 0))]

    # Blend overlay
    overlay = (1 - alpha) * out_image + alpha * heatmap_rgb
    overlay = np.clip(overlay, 0, 1)
    img_ax.set_data(overlay)
    fig.canvas.draw_idle()


# Sliders and UI
def setup_slider(ax_rect, label, valmin, valmax, valinit):
    return Slider(plt.axes(ax_rect), label, valmin, valmax, valinit=valinit)

slider_angle = setup_slider([0.25, 0.37, 0.65, 0.03], 'Angle', -180, 180, init_angle)
slider_scale_x = setup_slider([0.25, 0.32, 0.65, 0.03], 'Scale X', 0.1, 3.0, init_scale_x)
slider_scale_y = setup_slider([0.25, 0.27, 0.65, 0.03], 'Scale Y', 0.1, 3.0, init_scale_y)
slider_tx = setup_slider([0.25, 0.22, 0.65, 0.03], 'Move X', -200, 200, init_tx)
slider_ty = setup_slider([0.25, 0.17, 0.65, 0.03], 'Move Y', -200, 200, init_ty)
slider_alpha = setup_slider([0.25, 0.12, 0.65, 0.03], 'Alpha', 0, 1, init_alpha)

# Checkbox for uniform scaling
checkbox_ax = plt.axes([0.025, 0.12, 0.15, 0.1])
checkbox = CheckButtons(checkbox_ax, ['Uniform Scale'], [False])

# Update callback
def update(val=None):
    update_overlay(
        angle=slider_angle.val,
        scale_x=slider_scale_x.val,
        scale_y=slider_scale_y.val,
        tx=slider_tx.val,
        ty=slider_ty.val,
        alpha=slider_alpha.val,
        uniform=checkbox.get_status()[0]
    )

# Link sliders
for s in [slider_angle, slider_scale_x, slider_scale_y, slider_tx, slider_ty, slider_alpha]:
    s.on_changed(update)
checkbox.on_clicked(lambda label: update())

# Initial overlay
update()

plt.show()


# Use the annotator with your image and colormap
# Get the current overlay image
final_overlay = img_ax.get_array()

# Annotate using the overlay instead of raw image
annotator = PolygonAnnotator(final_overlay, cmap=None)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 800.55 → Matched m/z: 800.5515


MESA-LOADER: failed to open zink: /usr/lib/dri/zink_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)


Polygon drawn with 72 points.


In [9]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

/tmp/ipykernel_3314502/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


tissue
0    9380
1    7225
Name: count, dtype: int64


In [ ]:
# Save the modified AnnData object with the tissue mask
adata.write('processed_overlaid_h5ad_file_address.h5ad')
print("Saved modified AnnData with tissue mask.")

Saved modified AnnData with tissue mask.
